# Notebook 04: Fairness Evaluation

**EquiPay Canada**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

sys.path.insert(0, str(Path.cwd().parent))
from src.constants import COLS, GENDER_CODES
print('Setup complete')

In [ ]:
df = pd.read_csv('../data/processed/lfs_processed.csv')
gender_col = COLS.GENDER if COLS.GENDER in df.columns else 'SEX'
wage_col = COLS.HOURLY_EARNINGS if COLS.HOURLY_EARNINGS in df.columns else 'HRLYEARN'
df['IS_FEMALE'] = (df[gender_col] == 2).astype(int)
print(f'Loaded {len(df):,} records')

In [ ]:
features = [c for c in [COLS.EDUC, COLS.AGE_12, COLS.NOC_10, COLS.PROV] if c in df.columns]
df_clean = df[features + [wage_col, 'IS_FEMALE']].dropna()
X = df_clean[features]
y = df_clean[wage_col]
s = df_clean['IS_FEMALE']
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(X, y, s, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,}, Test: {len(X_test):,}')

In [ ]:
model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'R2: {r2_score(y_test, y_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}')

In [ ]:
male_mask = s_test == 0
female_mask = s_test == 1
male_pred = y_pred[male_mask].mean()
female_pred = y_pred[female_mask].mean()
di_ratio = female_pred / male_pred
rmse_m = np.sqrt(mean_squared_error(y_test[male_mask], y_pred[male_mask]))
rmse_f = np.sqrt(mean_squared_error(y_test[female_mask], y_pred[female_mask]))
print('FAIRNESS RESULTS')
print(f'Male pred mean: {male_pred:.2f}')
print(f'Female pred mean: {female_pred:.2f}')
print(f'Disparate Impact: {di_ratio:.3f}')
print(f'4/5ths Rule: {"PASS" if di_ratio >= 0.8 else "FAIL"}')
print(f'RMSE Male: {rmse_m:.2f}, Female: {rmse_f:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_pred[male_mask], bins=30, alpha=0.6, label='Male', color='steelblue')
ax.hist(y_pred[female_mask], bins=30, alpha=0.6, label='Female', color='coral')
ax.axvline(male_pred, color='steelblue', ls='--', lw=2)
ax.axvline(female_pred, color='coral', ls='--', lw=2)
ax.set_xlabel('Predicted Wage')
ax.set_ylabel('Count')
ax.set_title('Prediction Distribution by Gender')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print('=' * 50)
print('FAIRNESS SUMMARY')
print('=' * 50)
print(f'Disparate Impact Ratio: {di_ratio:.3f}')
print(f'4/5ths Rule: {"COMPLIANT" if di_ratio >= 0.8 else "NON-COMPLIANT"}')
print(f'Error Parity: {rmse_f/rmse_m:.3f}')
print('=' * 50)